# 02 — SQL Feature Engineering (DuckDB)

**Purpose:** Demonstrate SQL fluency on real retail data — CTEs, window functions,
RFM aggregations, partial-return detection. This is the Costco JD SQL signal.

**Each query is annotated clause-by-clause** — the interviewer is the audience.

---

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import duckdb
from src.features import load_raw

df = load_raw('../data/raw/online_retail_II.xlsx')
print(f'Loaded {len(df):,} rows')
df.head(2)

Loaded 1,067,371 rows


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,is_return
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,0
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,0


In [2]:
# Register DataFrame as a DuckDB view — no file write needed
con = duckdb.connect()
con.register('transactions', df)
print('DuckDB view registered: transactions')
con.execute('SELECT COUNT(*) AS n FROM transactions').df()

DuckDB view registered: transactions


,n
0,1067371


## Query 1 — Customer-level RFM via CTE

Recency, Frequency, Monetary value — the three dimensions that underpin customer segmentation.
The CTE separates the aggregation logic from the final select, making it readable and testable.

In [3]:
rfm_sql = """
WITH purchase_base AS (
    -- Limit to purchases only (exclude returns) and labeled customers
    SELECT
        customer_id,
        invoice_no,
        invoice_date,
        quantity * unit_price AS line_revenue
    FROM transactions
    WHERE is_return = 0
      AND customer_id IS NOT NULL
      AND unit_price > 0
      AND quantity > 0
),
rfm_raw AS (
    -- Aggregate to one row per customer
    SELECT
        customer_id,
        MAX(invoice_date)                        AS last_purchase_date,
        COUNT(DISTINCT invoice_no)               AS frequency,
        ROUND(SUM(line_revenue), 2)              AS monetary
    FROM purchase_base
    GROUP BY customer_id
),
snapshot_date AS (
    -- Single reference date: one day after last transaction in the dataset
    SELECT MAX(invoice_date) + INTERVAL '1 day' AS snap FROM transactions
)
SELECT
    r.customer_id,
    -- Recency: days since last purchase (lower = more recent = better)
    DATE_DIFF('day', r.last_purchase_date, s.snap)  AS recency_days,
    r.frequency,
    r.monetary,
    -- Quintile rankings (1=worst, 5=best for F and M; 1=best for R)
    NTILE(5) OVER (ORDER BY DATE_DIFF('day', r.last_purchase_date, s.snap) DESC) AS r_score,
    NTILE(5) OVER (ORDER BY r.frequency)          AS f_score,
    NTILE(5) OVER (ORDER BY r.monetary)           AS m_score
FROM rfm_raw r
CROSS JOIN snapshot_date s
ORDER BY monetary DESC
"""

rfm = con.execute(rfm_sql).df()
print(f'RFM computed for {len(rfm):,} customers')
print(f'\nRFM summary stats:')
print(rfm[['recency_days','frequency','monetary']].describe().round(1).to_string())
rfm.head(10)

RFM computed for 5,878 customers

RFM summary stats:
       recency_days  frequency  monetary
count        5878.0     5878.0    5878.0
mean          201.9        6.3    3018.6
std           209.4       13.0   14737.7
min             1.0        1.0       3.0
25%            26.0        1.0     348.8
50%            96.0        3.0     898.9
75%           380.0        7.0    2307.1
max           739.0      398.0  608821.6


,customer_id,recency_days,frequency,monetary,r_score,f_score,m_score
0,18102.0,1,145,608821.65,5,5,5
1,14646.0,2,151,528602.52,5,5,5
2,14156.0,10,156,313946.37,5,5,5
3,14911.0,2,398,295972.63,5,5,5
4,17450.0,9,51,246973.09,5,5,5
5,13694.0,4,143,196482.81,5,5,5
6,17511.0,3,60,175603.55,5,5,5
7,16446.0,1,2,168472.50,5,2,5
8,16684.0,5,55,147142.77,5,5,5
9,12415.0,25,28,144458.37,4,5,5


## Query 2 — Rank customers by lifetime return rate (window function)

Demonstrates `RANK()` and `DENSE_RANK()` over an ordered partition,
plus a ratio aggregation joining purchases and returns in a single query.

In [4]:
ranked_sql = """
WITH purchase_counts AS (
    SELECT customer_id, COUNT(DISTINCT invoice_no) AS n_purchases
    FROM transactions
    WHERE is_return = 0 AND customer_id IS NOT NULL
    GROUP BY customer_id
),
return_counts AS (
    SELECT customer_id, COUNT(DISTINCT invoice_no) AS n_returns,
           SUM(ABS(quantity * unit_price)) AS return_value
    FROM transactions
    WHERE is_return = 1 AND customer_id IS NOT NULL
    GROUP BY customer_id
),
combined AS (
    SELECT
        p.customer_id,
        p.n_purchases,
        COALESCE(r.n_returns, 0)      AS n_returns,
        COALESCE(r.return_value, 0.0) AS return_value,
        ROUND(
            COALESCE(r.n_returns, 0) * 1.0 / NULLIF(p.n_purchases, 0),
        4) AS return_rate
    FROM purchase_counts p
    LEFT JOIN return_counts r USING (customer_id)
)
SELECT
    customer_id,
    n_purchases,
    n_returns,
    ROUND(return_value, 2)       AS return_value_gbp,
    return_rate,
    -- RANK gives gaps for ties; DENSE_RANK does not
    RANK()        OVER (ORDER BY return_rate DESC, return_value DESC) AS rank_by_rate,
    DENSE_RANK()  OVER (ORDER BY return_rate DESC, return_value DESC) AS dense_rank,
    -- Percentile band for each customer
    NTILE(10) OVER (ORDER BY return_rate DESC) AS decile  -- 1 = highest returners
FROM combined
WHERE n_purchases >= 3   -- minimum 3 purchases for stable rate
ORDER BY rank_by_rate
"""

ranked = con.execute(ranked_sql).df()
print(f'Customers ranked: {len(ranked):,}')
print(f'Top-decile return rate range: {ranked[ranked.decile==1].return_rate.min():.1%} – {ranked[ranked.decile==1].return_rate.max():.1%}')
print('\nTop 15 by return rate:')
ranked.head(15)

Customers ranked: 3,311
Top-decile return rate range: 57.1% – 200.0%

Top 15 by return rate:


,customer_id,n_purchases,n_returns,return_value_gbp,return_rate,rank_by_rate,dense_rank,decile
0,14267.0,3,6,257.05,2.0000,1,1,1
1,17531.0,3,6,75.60,2.0000,2,2,1
2,17603.0,4,7,1673.47,1.7500,3,3,1
3,15202.0,3,5,8571.67,1.6667,4,4,1
4,12525.0,3,5,138.55,1.6667,5,5,1
5,12809.0,3,5,50.60,1.6667,6,6,1
6,13338.0,4,6,222.00,1.5000,7,7,1
7,15309.0,4,6,82.43,1.5000,8,8,1
8,16163.0,6,8,585.95,1.3333,9,9,1
9,15354.0,3,4,379.20,1.3333,10,10,1


## Query 3 — Return velocity: rolling 30-day window per customer

Demonstrates `SUM() OVER (PARTITION BY ... ORDER BY ... RANGE BETWEEN INTERVAL ...)` —
a rolling aggregate that's common in production feature pipelines.

In [5]:
velocity_sql = """
WITH return_events AS (
    SELECT
        customer_id,
        invoice_date,
        invoice_no,
        ABS(quantity * unit_price) AS return_value
    FROM transactions
    WHERE is_return = 1
      AND customer_id IS NOT NULL
),
rolling AS (
    SELECT
        customer_id,
        invoice_date,
        -- Count of return events in the preceding 30 days (inclusive)
        COUNT(*) OVER (
            PARTITION BY customer_id
            ORDER BY invoice_date
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        ) AS returns_last_30d,
        -- Return value in the preceding 30 days
        SUM(return_value) OVER (
            PARTITION BY customer_id
            ORDER BY invoice_date
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        ) AS return_value_last_30d
    FROM return_events
)
-- Customers whose rolling 30-day return count ever exceeded 3
SELECT DISTINCT customer_id,
       MAX(returns_last_30d)       AS peak_30d_return_count,
       ROUND(MAX(return_value_last_30d), 2) AS peak_30d_return_value
FROM rolling
GROUP BY customer_id
HAVING MAX(returns_last_30d) >= 3
ORDER BY peak_30d_return_count DESC, peak_30d_return_value DESC
"""

velocity = con.execute(velocity_sql).df()
print(f'Customers with ≥3 returns in any 30-day window: {len(velocity):,}')
velocity.head(10)

Customers with ≥3 returns in any 30-day window: 1,139


,customer_id,peak_30d_return_count,peak_30d_return_value
0,12607.0,101,1579.51
1,13091.0,74,2426.04
2,14911.0,71,6691.83
3,14934.0,62,1057.30
4,12415.0,57,436.73
5,16555.0,49,1629.79
6,14277.0,45,11880.84
7,16801.0,45,751.90
8,14410.0,45,271.92
9,15369.0,44,2945.38


## Query 4 — Partial-return detection

A partial return is when a customer returns *some but not all* items from an order.
These are a stronger wardrobing signal than a full return.
Requires joining orders to their corresponding cancellations.

In [6]:
partial_sql = """
WITH order_totals AS (
    -- Total quantity and revenue per invoice (purchases only)
    SELECT
        invoice_no,
        customer_id,
        invoice_date,
        SUM(quantity)               AS total_qty,
        SUM(quantity * unit_price)  AS total_revenue
    FROM transactions
    WHERE is_return = 0
      AND customer_id IS NOT NULL
    GROUP BY invoice_no, customer_id, invoice_date
),
return_totals AS (
    -- Total return quantity per original invoice
    -- Return invoices start with 'C'; the digits following 'C' reference the original
    SELECT
        customer_id,
        invoice_date                    AS return_date,
        SUM(ABS(quantity))              AS returned_qty,
        SUM(ABS(quantity * unit_price)) AS returned_value
    FROM transactions
    WHERE is_return = 1
      AND customer_id IS NOT NULL
    GROUP BY customer_id, invoice_date
),
partial_returns AS (
    -- Match returns to orders by customer + date proximity (within 60 days)
    SELECT
        o.invoice_no,
        o.customer_id,
        o.invoice_date  AS order_date,
        r.return_date,
        DATE_DIFF('day', o.invoice_date, r.return_date) AS days_to_return,
        o.total_qty,
        r.returned_qty,
        ROUND(r.returned_qty * 1.0 / NULLIF(o.total_qty, 0), 4) AS return_qty_ratio,
        ROUND(r.returned_value, 2) AS returned_value
    FROM order_totals o
    JOIN return_totals r
      ON  o.customer_id = r.customer_id
      AND DATE_DIFF('day', o.invoice_date, r.return_date) BETWEEN 0 AND 60
)
SELECT *,
    CASE
        WHEN return_qty_ratio > 0 AND return_qty_ratio < 1 THEN 'Partial'
        WHEN return_qty_ratio >= 1                          THEN 'Full'
        ELSE 'None'
    END AS return_type
FROM partial_returns
WHERE returned_qty > 0
ORDER BY returned_value DESC
"""

partial = con.execute(partial_sql).df()
type_counts = partial['return_type'].value_counts()
print('Return type breakdown:')
print(type_counts.to_string())
print(f'\nPartial returns: {type_counts.get("Partial",0):,}')
print(f'Full returns:    {type_counts.get("Full",0):,}')
print('\nTop 10 highest-value matched returns:')
partial.head(10)

Return type breakdown:
return_type
Partial    26572
Full        3093
None           3

Partial returns: 26,572
Full returns:    3,093

Top 10 highest-value matched returns:


,invoice_no,customer_id,order_date,return_date,days_to_return,total_qty,returned_qty,return_qty_ratio,returned_value,return_type
0,581483,16446.0,2011-12-09 09:15:00,2011-12-09 09:27:00,0,80995.0,80995.0,1.0000,168469.60,Full
1,541431,12346.0,2011-01-18 10:01:00,2011-01-18 10:17:00,0,74215.0,74215.0,1.0000,77183.60,Full
2,556444,15098.0,2011-06-10 15:28:00,2011-06-10 15:31:00,0,60.0,1.0,0.0167,38970.00,Partial
3,556446,15098.0,2011-06-10 15:33:00,2011-06-10 15:31:00,0,1.0,1.0,1.0000,38970.00,Full
4,556442,15098.0,2011-06-10 15:22:00,2011-06-10 15:31:00,0,60.0,1.0,0.0167,38970.00,Partial
5,550461,15749.0,2011-04-18 13:20:00,2011-04-18 13:08:00,0,9014.0,9014.0,1.0000,22998.40,Full
6,518505,14277.0,2010-08-09 13:10:00,2010-09-28 11:02:00,50,87167.0,87167.0,1.0000,11880.84,Full
7,563562,16029.0,2011-08-17 14:02:00,2011-10-11 11:10:00,55,412.0,6480.0,15.7282,11816.64,Full
8,564327,16029.0,2011-08-24 13:33:00,2011-10-11 11:10:00,48,658.0,6480.0,9.8480,11816.64,Full
9,569649,16029.0,2011-10-05 12:42:00,2011-10-11 11:10:00,6,324.0,6480.0,20.0000,11816.64,Full


## Query 5 — Category return rate with seasonal breakdown

Product-level return rates stratified by quarter — useful for detecting seasonal
wardrobing patterns (e.g. holiday purchases returned in January).

In [7]:
seasonal_sql = """
WITH product_quarter AS (
    SELECT
        stock_code,
        description,
        EXTRACT(QUARTER FROM invoice_date)::INT AS quarter,
        EXTRACT(YEAR FROM invoice_date)::INT    AS year,
        is_return,
        COUNT(*) AS n_rows
    FROM transactions
    WHERE unit_price > 0
    GROUP BY stock_code, description, quarter, year, is_return
),
pivoted AS (
    SELECT
        stock_code,
        MAX(description) AS description,
        year,
        quarter,
        SUM(CASE WHEN is_return = 0 THEN n_rows ELSE 0 END) AS purchases,
        SUM(CASE WHEN is_return = 1 THEN n_rows ELSE 0 END) AS returns
    FROM product_quarter
    GROUP BY stock_code, year, quarter
)
SELECT
    stock_code,
    description,
    year,
    quarter,
    purchases,
    returns,
    ROUND(returns * 1.0 / NULLIF(purchases, 0), 4) AS return_rate,
    -- Flag Q4 (holiday) vs. Q1 (post-holiday return season)
    CASE WHEN quarter = 4 THEN 'Holiday' WHEN quarter = 1 THEN 'Post-holiday' ELSE 'Other' END AS season
FROM pivoted
WHERE purchases >= 20
ORDER BY return_rate DESC, purchases DESC
"""

seasonal = con.execute(seasonal_sql).df()
print(f'Product-quarter rows: {len(seasonal):,}')
print('\nAvg return rate by season:')
print(seasonal.groupby('season')['return_rate'].mean().round(4).to_string())
print('\nHighest return-rate products (min 20 purchases):')
seasonal.head(10)

Product-quarter rows: 12,335

Avg return rate by season:
season
Holiday         0.0163
Other           0.0191
Post-holiday    0.0186

Highest return-rate products (min 20 purchases):


,stock_code,description,year,quarter,purchases,returns,return_rate,season
0,79323W,WHITE CHERRY LIGHTS,2010,2,25.0,104.0,4.1600,Other
1,79323P,PINK CHERRY LIGHTS,2010,2,20.0,62.0,3.1000,Other
2,M,Manual,2011,1,58.0,55.0,0.9483,Post-holiday
3,M,Manual,2011,3,71.0,67.0,0.9437,Other
4,M,Manual,2011,2,62.0,57.0,0.9194,Other
5,ADJUST,Adjustment by john on 26/01/2010 17,2010,1,36.0,28.0,0.7778,Post-holiday
6,M,Manual,2010,3,115.0,73.0,0.6348,Other
7,M,Manual,2011,4,98.0,58.0,0.5918,Holiday
8,M,Manual,2010,1,129.0,73.0,0.5659,Post-holiday
9,M,Manual,2010,2,139.0,76.0,0.5468,Other


## Export SQL-derived features

In [8]:
import os
os.makedirs('../data/processed', exist_ok=True)

rfm.to_parquet('../data/processed/rfm_features.parquet', index=False)
ranked.to_parquet('../data/processed/customer_return_ranks.parquet', index=False)
velocity.to_parquet('../data/processed/return_velocity.parquet', index=False)

print('Exported to data/processed/:')
print('  rfm_features.parquet           — RFM scores for all customers')
print('  customer_return_ranks.parquet  — return rate + decile rankings')
print('  return_velocity.parquet        — customers with burst return activity')
print()
print(f'RFM: {len(rfm):,} customers')
print(f'Ranked: {len(ranked):,} customers (≥3 purchases)')
print(f'High-velocity: {len(velocity):,} customers (≥3 returns in 30d)')

con.close()

Exported to data/processed/:
  rfm_features.parquet           — RFM scores for all customers
  customer_return_ranks.parquet  — return rate + decile rankings
  return_velocity.parquet        — customers with burst return activity

RFM: 5,878 customers
Ranked: 3,311 customers (≥3 purchases)
High-velocity: 1,139 customers (≥3 returns in 30d)
